# Module 6: worked practice solutions

## Goal and route

Use the module's foundation, concepts and guided page first. This notebook supports the existing outcomes; it does not replace the complete study. This file contains answers: compare only after attempting the practice notebook.

Small check inputs are authored teaching examples. The final sandbox uses attributed real development records. Neither is a fresh final evaluation. Keep this notebook inside the complete source checkout/archive.


## Setup

Use the existing core environment with regression, visualization and verification extras for Modules 5–6; use the separate framework kernel for Module 8. No package installs or account tokens belong in this notebook.

In [ ]:
import sys
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"curriculum/modules.json").is_file())
sys.path.insert(0, str(root/"src"))
from pathlib import Path
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "2")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "2")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from practical_ml_ai.practice import check
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "curriculum/modules.json").is_file())
results = {}
plt.rcParams.update({"font.size": 12, "axes.spines.top": False, "axes.spines.right": False})
print("Core environment" if 6 != 8 else "Framework environment with TensorFlow; CPU is sufficient")

## Task 1: confusion_counts

Return integer tp, fp, fn, tn keys. Labels are binary 0/1; predict positive when probability >= threshold. Inputs are equal-length nonempty sequences.

**Predict and explain:** Trace the row whose score equals the threshold. Count all four outcomes; their sum must equal the number of rows.

Write your prediction, attempted approach and what you changed here before running the check.

In [ ]:
def confusion_counts(actual, probability, threshold):
    actual = np.asarray(actual)
    positive = np.asarray(probability) >= threshold
    return {"tp": int(((actual == 1) & positive).sum()), "fp": int(((actual == 0) & positive).sum()),
            "fn": int(((actual == 1) & ~positive).sum()), "tn": int(((actual == 0) & ~positive).sum())}

In [ ]:
results["m06_confusion"] = check("m06_confusion", confusion_counts)

## Task 2: choose_threshold

Using validation labels only, return the threshold minimizing miss_cost*FN + false_alarm_cost*FP; break ties with the smallest threshold. Use confusion_counts.

**Predict and explain:** Write a cost table before selecting. A threshold preferred under 3:1 may lose under 1:5. These are assumed cost units, not measured dollars.

Write your prediction, attempted approach and what you changed here before running the check.

In [ ]:
def choose_threshold(actual, probability, thresholds, miss_cost, false_alarm_cost):
    candidates = []
    for threshold in thresholds:
        counts = confusion_counts(actual, probability, threshold)
        cost = miss_cost * counts["fn"] + false_alarm_cost * counts["fp"]
        candidates.append((cost, threshold))
    return min(candidates)[1]

In [ ]:
results["m06_threshold"] = check("m06_threshold", choose_threshold)

## Task 3: fit_classifier

Return (fitted Pipeline, one-dimensional positive-class probabilities). Name StandardScaler scaler and LogisticRegression model. Use max_iter=1000, random_state=42; fit only on training.

**Predict and explain:** A probability is different from a class. Inspect the fitted class order before choosing the probability column.

Write your prediction, attempted approach and what you changed here before running the check.

In [ ]:
def fit_classifier(train_x, train_y, new_x):
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    model = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000, random_state=42))])
    model.fit(train_x, train_y)
    positive_column = list(model.classes_).index(1)
    return model, model.predict_proba(new_x)[:, positive_column]

In [ ]:
results["m06_pipeline"] = check("m06_pipeline", fit_classifier)

## Apply the functions to real development records

This supplied scaffold uses your checked functions. If a check needs work, return to its function and rerun from the setup cell. A green local check supports these cases, not every possible implementation or your explanation.

Before running, write which inputs and labels are available and which partition may select a decision. Compare the exact tables with the figure; no chart-reading requirement substitutes for the numeric evidence.

In [ ]:
import sys
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"curriculum/modules.json").is_file())
sys.path.insert(0, str(root/"src"))
demo_verified = False
if len(results) == 3 and all(item["state"] == "passed" for item in results.values()):
    from practical_ml_ai.continuation_data import wine_splits
    split, features, provenance = wine_splits()
    training, validation = split["train"], split["validation"]
    # Final-test labels are not used in this sandbox.
    model, probability = fit_classifier(training[features], training.label, validation[features])
    thresholds = [.3, .5, .7]
    threshold = choose_threshold(validation.label, probability, thresholds, 3, 1)
    prior_probability = np.full(len(validation), training.label.mean())
    prior_threshold = choose_threshold(validation.label, prior_probability, thresholds, 3, 1)
    rows = []
    for method, scores, chosen in [("logistic", probability, threshold), ("training_prior", prior_probability, prior_threshold)]:
        counts = confusion_counts(validation.label, scores, chosen)
        rows.append({"method": method, "threshold": chosen, **counts, "assumed_cost": 3*counts["fn"] + counts["fp"]})
    display(pd.DataFrame(rows))
    tradeoffs = pd.DataFrame([{"threshold": t, **confusion_counts(validation.label, probability, t)} for t in thresholds])
    display(tradeoffs)
    ax = tradeoffs.set_index("threshold")[["fn", "fp"]].plot.bar(figsize=(7, 4), color=["#17658b", "#a65c00"], rot=0)
    ax.set(title="Validation threshold tradeoff", xlabel="Decision threshold", ylabel="Number of records")
    for patch in ax.patches[len(tradeoffs):]:
        patch.set_hatch("//")
    ax.legend(["Missed positives (FN)", "False alarms (FP)"]); plt.tight_layout(); plt.show()
    demo_verified = sum(rows[0][key] for key in ["tp", "fp", "fn", "tn"]) == len(validation)
    print("Source: UCI Wine Quality, DOI 10.24432/C56S3T; grouped development partitions from the course source contract.")
    print("Positive means rating >=7; the 3:1 error-cost ratio is an illustrative assumption, not an IT incident cost estimate.")
else:
    print("Development sandbox not run: finish the three functions and rerun from Setup.")

## Independent transfer and handoff

Return to the module practice page for the changed IT task, diagnosis and decision memo. Retain your code, predictions, feedback, one manually checked number and a short explanation of a failure. Then run the complete module walkthrough for its full baseline/final-evaluation/artifact contract.

An unattempted starter executes with visible `not_attempted` feedback and skips the sandbox; that is not completed learning. Solutions demonstrate a worked approach, not evidence that a learner can reproduce it independently. Human piloting is pending.
